# Why the pixel sum never converged — slide figures

Standalone figures for the group talk of 2026-10-20. Last time this group saw
the diffuse-sky Faraday signal computed as a sum over HEALPix pixels; the
2026-08-18 audit (tag `audit-2026-08-18`) showed that sum does not converge.
This notebook tells that story in plots.

It depends only on numpy/healpy/h5py/astropy/matplotlib and the input maps in
`data/` — nothing from `src/` or `scripts/`. PNGs are written to
`notebooks/slides_pixel_sum/`, which is git-ignored; rerun the notebook to
regenerate them.

In [ ]:
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
DATA = REPO / "data"
OUT = REPO / "notebooks" / "slides_pixel_sum"
OUT.mkdir(exist_ok=True)

C = 299792458.0  # m/s


def lam2(freq_mhz):
    """Wavelength squared in m^2."""
    return (C / (np.asarray(freq_mhz) * 1e6)) ** 2


# Slides are 16:9; draw at the size the figure is shown so fonts read.
plt.rcParams.update(
    {
        "figure.figsize": (8.0, 4.5),
        "figure.dpi": 110,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.size": 14,
        "axes.titlesize": 15,
        "axes.labelsize": 14,
        "legend.fontsize": 12,
    }
)


def save(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path)
    print("wrote", path.relative_to(REPO))

## Inputs

- `faraday2020v2.hdf5` — Hutschenreuter et al. (2022) Faraday sky, RING,
  nside 512, rad m$^{-2}$ (posterior mean and std).
- `wmap_band_iqumap_r9_9yr_K_v5.fits` — WMAP 9-yr K band, NESTED → RING,
  mK. Q/U are healpy/COSMO.
- `haslam408_dsds_Remazeilles2014.fits` — 408 MHz, RING, K.

In [ ]:
with h5py.File(DATA / "faraday2020v2.hdf5", "r") as f:
    RM = f["faraday_sky_mean"][:]
    RM_STD = f["faraday_sky_std"][:]

# WMAP K band: mK thermodynamic -> K Rayleigh-Jeans at 23 GHz
T_CMB = 2.7255
x = 6.62607015e-34 * 23e9 / (1.380649e-23 * T_CMB)
TH_TO_RJ = x**2 * np.exp(x) / np.expm1(x) ** 2
with fits.open(DATA / "wmap_band_iqumap_r9_9yr_K_v5.fits") as h:
    d = h["Stokes Maps"].data
    Q23 = hp.reorder(d["Q_POLARISATION"].astype(np.float64), n2r=True)
    U23 = hp.reorder(d["U_POLARISATION"].astype(np.float64), n2r=True)
Q23 *= 1e-3 * TH_TO_RJ
U23 *= 1e-3 * TH_TO_RJ

BETA_QU = -2.8  # config.BETA_QU


def sky_qu(freq_mhz):
    """WMAP K Q, U power-law scaled to freq_mhz, K_RJ, healpy/COSMO."""
    s = (freq_mhz / 23e3) ** BETA_QU
    return Q23 * s, U23 * s


NSIDE = hp.npix2nside(RM.size)
assert hp.npix2nside(Q23.size) == NSIDE  # common native grid, never degraded
print("nside", NSIDE, "| RM rms", RM.std(), "rad/m^2")

## 1. One source at one Faraday depth: the rotation is spectrally resolved

The Faraday phase of $P = Q + iU$ is $2\,\mathrm{RM}\,\lambda^2$. A channel of
width $\Delta\nu$ spans $\delta\lambda^2 = 2\lambda^2\,\Delta\nu/\nu$, so the
phase advances by $2\,\mathrm{RM}\,\delta\lambda^2$ per channel. The
rotation is sampled without aliasing while that step is below $\pi$, i.e.
for $|\mathrm{RM}| < \pi / (2\,\delta\lambda^2)$.

Channels: parent 25 kHz; zoom 25 kHz / 64 = 390.625 Hz
(`channelization.ZOOM_STEP_HZ`).

In [ ]:
PARENT_HZ = 25e3
ZOOM_HZ = PARENT_HZ / 64
RM_SRC = 100.0  # rad/m^2
BANDS_MHZ = (10.0, 30.0, 50.0)

print(f"{'nu':>4} {'chan':>6} {'dlam2 [m^2]':>12} "
      f"{'rad/chan @RM=100':>17} {'RM_max':>8}")
for f0 in BANDS_MHZ:
    for name, dnu in (("parent", PARENT_HZ), ("zoom", ZOOM_HZ)):
        dl2 = 2 * lam2(f0) * dnu / (f0 * 1e6)
        print(f"{f0:4.0f} {name:>6} {dl2:12.3e} "
              f"{2 * RM_SRC * dl2:17.3f} {np.pi / (2 * dl2):8.0f}")
    period = np.pi * f0 * 1e6 / (2 * RM_SRC * lam2(f0))
    print(f"     one turn of P at RM=100: {period:.0f} Hz")

In [ ]:
def channel_average(f0_mhz, rm, width_hz, n_chan, n_sub=64):
    """Top-hat channel means of exp(2i rm lam^2), centred on f0.

    Returns channel centres (Hz offset from f0) and the complex means.
    The exact lam^2(nu) is sampled at n_sub points per channel.
    """
    k = np.arange(n_chan) - n_chan // 2
    sub = (np.arange(n_sub) + 0.5) / n_sub - 0.5
    off = (k[:, None] + sub[None, :]) * width_hz
    p = np.exp(2j * rm * lam2(f0_mhz + off * 1e-6))
    return k * width_hz, p.mean(axis=1)


ZOOM_BANDS_MHZ = (30.0, 50.0)
C_Q, C_U = "#0072B2", "#D55E00"

# Q and U on the same axes, one parent channel (64 zoom bins) wide
fig, axes = plt.subplots(
    1, 2, figsize=(10.0, 3.8), sharey=True, constrained_layout=True
)
for ax, f0 in zip(axes, ZOOM_BANDS_MHZ):
    x_fine, p_fine = channel_average(f0, RM_SRC, ZOOM_HZ / 32, 64 * 32,
                                     n_sub=1)
    x_zoom, p_zoom = channel_average(f0, RM_SRC, ZOOM_HZ, 64)
    for part, col, lab in ((np.real, C_Q, "Q"), (np.imag, C_U, "U")):
        ax.plot(x_fine / 1e3, part(p_fine), color=col, lw=0.8, alpha=0.45)
        ax.plot(x_zoom / 1e3, part(p_zoom), ".", color=col, ms=6,
                label=lab)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(-PARENT_HZ / 2e3, PARENT_HZ / 2e3)
    dl2 = 2 * lam2(f0) * ZOOM_HZ / (f0 * 1e6)
    ax.set_title(f"{f0:.0f} MHz:  {2 * RM_SRC * dl2:.2f} rad / zoom bin")
    ax.set_xlabel(f"$\\nu - {f0:.0f}$ MHz  [kHz]")
axes[0].set_ylabel("fraction of $|P_0|$")
fig.legend(*axes[1].get_legend_handles_labels(),
           loc="outside right center")
save(fig, "01_single_source_rm100")

**Figure 1.** Fractional $Q$ and $U$ of one linearly polarized source,
$\mathrm{RM} = 100$ rad m$^{-2}$, seen by an ideal calibrated zenith
polarimeter (no beam, no leakage), across one 25 kHz parent channel at 30
and 50 MHz. Dots: means over each of the 64 zoom bins (ideal 390.6 Hz
top-hats, not the PFB response); lines: the unbinned rotation
$P_0\,e^{2i\,\mathrm{RM}\,\lambda^2}$. The zoom bins follow the rotation
(0.52 and 0.11 rad per bin; five turns and about one across the parent
channel), and $U$ lagging $Q$ by a quarter turn shows the angle rotating.
The parent channel alone would average $P$ to near zero. Below ~14 MHz the
zoom bins no longer resolve RM = 100 (see the table above).

## 2. The previous approach: rotate every pixel, then sum

The diffuse sky was rotated pixel by pixel,
$(Q + iU)_\mathrm{rot}(\hat n) = (Q + iU)(\hat n)\,
e^{2i\,\mathrm{RM}(\hat n)\,\lambda^2}$, at native nside 512, and then
summed against the beam.

In [ ]:
from matplotlib.colors import Normalize, SymLogNorm

F_MAP = 30.0
KK = 1e3  # plot unit, K
NORMS = {
    "linear": Normalize(vmin=-20.0, vmax=20.0),
    # the scale of the earlier figure (paper_plots.ipynb, cell 20)
    "symlog": SymLogNorm(linthresh=1.0, linscale=1, vmin=-600.0,
                         vmax=600.0, base=10),
}


def qu_rot_maps(freq_mhz, nside=NSIDE):
    """Intrinsic and rotated (Q, U) at freq_mhz.

    nside < native reproduces the earlier figure: Q, U and RM are
    ud_graded first and the rotation applied on the coarse grid.
    """
    q, u = sky_qu(freq_mhz)
    rm = RM
    if nside != NSIDE:
        q, u, rm = (hp.ud_grade(m, nside) for m in (q, u, rm))
    p_rot = (q + 1j * u) * np.exp(2j * rm * lam2(freq_mhz))
    return q, u, p_rot.real, p_rot.imag


def moll_image(m, xsize=1000):
    """Galactic Mollweide raster of a RING map (longitude increasing left)."""
    proj = hp.projector.MollweideProj(xsize=xsize, flipconv="astro")
    ns = hp.npix2nside(m.size)
    img = proj.projmap(m, lambda x, y, z: hp.vec2pix(ns, x, y, z))
    return np.ma.masked_invalid(img), proj.get_extent()


def plot_qu_maps(nside=NSIDE, scale="linear"):
    q, u, qr, ur = qu_rot_maps(F_MAP, nside)
    norm = NORMS[scale]
    panels = {
        (0, 0): (q, "$Q$"),
        (1, 0): (u, "$U$"),
        (0, 1): (qr, f"$Q$ rotated, {F_MAP:.0f} MHz"),
        (1, 1): (ur, f"$U$ rotated, {F_MAP:.0f} MHz"),
    }
    # Columns sit GAP_IN apart, leaving room for arrows drawn on the slide.
    W_IN, H_IN, MAP_W_IN, GAP_IN = 12.0, 6.4, 4.6, 2.0
    MAP_H_IN = MAP_W_IN / 2
    fig = plt.figure(figsize=(W_IN, H_IN))
    x0 = (W_IN - 2 * MAP_W_IN - GAP_IN) / 2
    t = np.linspace(0, 2 * np.pi, 400)
    for (r, c), (m, title) in panels.items():
        left = x0 + c * (MAP_W_IN + GAP_IN)
        bottom = H_IN - 0.45 - (r + 1) * MAP_H_IN - r * 0.45
        ax = fig.add_axes([left / W_IN, bottom / H_IN,
                           MAP_W_IN / W_IN, MAP_H_IN / H_IN])
        img, ext = moll_image(m / KK)
        # ~one raster pixel per printed pixel, nearest: no colour blending
        ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r",
                  norm=norm, interpolation="nearest")
        ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
        ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
        ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
        ax.set_axis_off()
        ax.set_title(title)
    cax = fig.add_axes([0.3, 0.2 / H_IN, 0.4, 0.18 / H_IN])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="RdBu_r"),
                      cax=cax, orientation="horizontal")
    cb.set_label(f"brightness temperature at {F_MAP:.0f} MHz  [$10^3$ K]")
    return fig


save(plot_qu_maps(), "02_qu_maps_rotated")

**Figure 2.** Galactic Mollweide maps of the diffuse polarized sky at
30 MHz: WMAP 9-yr K-band $Q$, $U$ (K$_\mathrm{RJ}$, healpy/COSMO) scaled with
$\beta = -2.8$, before (left) and after (right) multiplying each native
nside-512 pixel by $e^{2i\,\mathrm{RM}\,\lambda^2}$ with the Hutschenreuter et
al. (2022) posterior-mean RM. Linear colour scale, $\pm 2\times10^4$ K,
shared by all panels. The intrinsic maps show the large-scale structure
(Galactic plane, North Polar Spur) under WMAP's per-pixel noise, which
dominates at high latitude. After rotation the large-scale structure is gone:
the phase changes by ~1700 rad between neighbouring pixels, so neighbouring
pixels have unrelated signs. Only $|P|$ survives, which is why the plane is
still visible as a band of larger scatter. The projection resamples the
nside-512 maps, so the exact speckle pattern is a display artefact; the loss
of structure is not.

## 3. The bug: the pixel grid aliases the Faraday phase

The phase applied to each pixel is $2\lambda^2\,\mathrm{RM}$: the RM map
times 1800 rad (10 MHz), 200 rad (30 MHz) or 72 rad (50 MHz) per rad m$^{-2}$.
The same Nyquist test as Fig. 1 applies along the sky: the phase must change
by less than $\pi$ between samples, here between neighbouring pixels.

In [ ]:
# Phase step between neighbouring pixels, over the whole sky
nbr = hp.get_all_neighbours(NSIDE, np.arange(RM.size))
d_rm = np.where(nbr >= 0, np.abs(RM[nbr] - RM[None]), np.nan)
step_rm = np.nanmean(d_rm, axis=0)  # mean |dRM| to the 8 neighbours
print("mean phase step to neighbouring pixels [rad]")
print(f"{'MHz':>6} {'p10':>7} {'p50':>7} {'p90':>7}")
for f0 in (1400.0, 408.0, 50.0, 30.0, 10.0):
    p = 2 * lam2(f0) * np.percentile(step_rm, (10, 50, 90))
    print(f"{f0:6.0f} {p[0]:7.2f} {p[1]:7.2f} {p[2]:7.1f}")
del nbr, d_rm

In [ ]:
RM_NORM = Normalize(vmin=-150, vmax=150)  # shared with Fig. 4b

fig = plt.figure(figsize=(8.0, 4.6))
ax = fig.add_axes([0.02, 0.2, 0.96, 0.75])
img, ext = moll_image(RM)
ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r", norm=RM_NORM,
          interpolation="nearest")
t = np.linspace(0, 2 * np.pi, 400)
ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
ax.set_axis_off()
cax = fig.add_axes([0.3, 0.1, 0.4, 0.035])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=RM_NORM, cmap="RdBu_r"),
                  cax=cax, orientation="horizontal")
cb.set_label("RM  [rad m$^{-2}$]")
save(fig, "03_rm_map")

**Figure 3.** Hutschenreuter et al. (2022) posterior-mean RM, Galactic
Mollweide, native nside 512, linear scale clipped at $\pm150$ rad m$^{-2}$.
The phase applied per pixel is this map times $2\lambda^2$: ×1800 at 10 MHz,
×200 at 30 MHz, ×72 at 50 MHz.

In [ ]:
# (a) 1-D cut: the pixel samples of the phase along a 2-degree line
CUT_L, CUT_B, CUT_LEN = 30.0, 0.0, 2.0  # deg; vertical, across the plane
CUT_FREQS = (1400.0, 408.0, 50.0, 30.0)
n_fine = 400_000
s = np.linspace(-CUT_LEN / 2, CUT_LEN / 2, n_fine)  # deg along the cut
pix = hp.ang2pix(NSIDE, CUT_L + 0 * s, CUT_B + s, lonlat=True)
edges = np.flatnonzero(np.r_[True, pix[1:] != pix[:-1], True])
s_pix = np.array([s[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])
rm_pix = RM[pix[edges[:-1]]]
rm_line = np.interp(s, s_pix, rm_pix)  # smooth stand-in between centres
print(len(s_pix), "pixels along the cut;",
      f"RM {rm_pix.min():.0f} .. {rm_pix.max():.0f} rad/m^2")

fig, axes = plt.subplots(
    len(CUT_FREQS) + 1, 1, figsize=(10.0, 8.0), sharex=True,
    constrained_layout=True,
)
x_arc, xp_arc = s * 60, s_pix * 60
axes[0].plot(x_arc, rm_line, color="0.4", lw=1)
axes[0].plot(xp_arc, rm_pix, "o", color="k", ms=4)
axes[0].set_ylabel("RM\n[rad m$^{-2}$]")
for ax, f0 in zip(axes[1:], CUT_FREQS):
    ax.plot(x_arc, np.cos(2 * lam2(f0) * rm_line), color="0.55",
            lw=0.4)
    ax.plot(xp_arc, np.cos(2 * lam2(f0) * rm_pix), "o", color=C_Q, ms=4)
    step = 2 * lam2(f0) * np.median(np.abs(np.diff(rm_pix)))
    lab = f"{f0/1e3:.1f} GHz" if f0 >= 1e3 else f"{f0:.0f} MHz"
    ax.set_ylabel(f"$Q/|P_0|$\n{lab}")
    step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
    ax.set_title(f"{lab}:  {step_txt} rad per pixel", fontsize=12,
                 loc="right", pad=2)
    ax.set_ylim(-1.2, 1.2)
axes[-1].set_xlabel(
    f"position along the cut, $b$ at $l = {CUT_L:.0f}^\\circ$  [arcmin]"
)
save(fig, "04a_cut_phase_sampling")

**Figure 4a.** Faraday rotation sampled along a $2^\circ$ cut across the
Galactic plane at $l = 30^\circ$. Top: RM at the centres of the
nside-512 pixels the cut crosses (dots, one value per pixel, as the
simulation uses them) and linearly interpolated between them (line, a
stand-in for the smooth sky). Below: $Q/|P_0| = \cos(2\lambda^2\,\mathrm{RM})$
for a source of intrinsic angle 0, at the pixels (dots) and along the line.
Titles give the median phase step between consecutive pixels. At 1.4 GHz the
pixels follow the rotation; at 408 MHz the rotation turns more than once
between pixels and the dots no longer trace it; at 50 and 30 MHz each pixel
holds tens to hundreds of turns, so the pixel values are effectively random
phases. This cut lies in the plane, where gradients are largest; over the
whole sky the median step at 30 MHz is ~130 rad and the 10th percentile ~31
rad, all far above $\pi$.

In [ ]:
# (b) 5-degree patch of the wrapped phase, pixels drawn as they are.
# Off the plane, with |RM| < 150 so the RM panel shares Fig. 3's scale.
PATCH_L, PATCH_B, PATCH_DEG = 250.0, -15.0, 5.0
PATCH_FREQS = (1400.0, 408.0, 50.0, 30.0)
RESO = 0.5  # arcmin per raster pixel; an nside-512 pixel is ~6.9'
gproj = hp.projector.GnomonicProj(
    rot=(PATCH_L, PATCH_B), xsize=int(PATCH_DEG * 60 / RESO), reso=RESO
)


def patch(m):
    return gproj.projmap(m, lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))


in_patch = hp.query_disc(
    NSIDE, hp.ang2vec(PATCH_L, PATCH_B, lonlat=True),
    np.radians(PATCH_DEG / 2),
)
print(f"RM in patch: {RM[in_patch].min():.0f} .. {RM[in_patch].max():.0f}; "
      f"median step {np.median(step_rm[in_patch]):.2f} rad/m^2 "
      f"(sky median {np.median(step_rm):.2f})")

gext = gproj.get_extent()
fig, axes = plt.subplots(1, 5, figsize=(13.0, 3.6),
                         constrained_layout=True)
axes[0].imshow(patch(RM), origin="lower", extent=gext, cmap="RdBu_r",
               norm=RM_NORM, interpolation="nearest")
axes[0].set_title("RM")
fig.colorbar(plt.cm.ScalarMappable(norm=RM_NORM, cmap="RdBu_r"),
             ax=axes[0], orientation="horizontal", fraction=0.06, pad=0.04,
             label="RM  [rad m$^{-2}$]")
for ax, f0 in zip(axes[1:], PATCH_FREQS):
    ph = np.angle(np.exp(2j * lam2(f0) * patch(RM)))
    im = ax.imshow(ph, origin="lower", extent=gext, cmap="twilight",
                   vmin=-np.pi, vmax=np.pi, interpolation="nearest")
    lab = f"{f0/1e3:.1f} GHz" if f0 >= 1e3 else f"{f0:.0f} MHz"
    step = 2 * lam2(f0) * np.median(step_rm[in_patch])
    step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
    ax.set_title(f"{lab}: {step_txt} rad/pixel", fontsize=13)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
cb = fig.colorbar(im, ax=axes[1:], orientation="horizontal", fraction=0.06,
                  pad=0.04, aspect=60)
cb.set_ticks([-np.pi, 0, np.pi], labels=["$-\\pi$", "0", "$\\pi$"])
cb.set_label("phase $2\\lambda^2\\,\\mathrm{RM}$, wrapped to $[-\\pi, \\pi)$")
save(fig, "04b_patch_phase_wrapped")

**Figure 4b.** A $5^\circ \times 5^\circ$ gnomonic patch centred on
$(l, b) = (250^\circ, -15^\circ)$, off the Galactic plane, drawn at 0.5' per
screen pixel so each nside-512 pixel (~6.9') shows as a tile. Left: RM, on
the colour scale of Fig. 3. Right: the phase $2\lambda^2\,\mathrm{RM}$ wrapped
to $[-\pi,\pi)$ (cyclic colour map); titles give the median phase step
between neighbouring pixels in the patch. The patch's median RM step
(1.65 rad m$^{-2}$) is above the sky median (0.65), but the sky median at
30 MHz is still 130 rad per pixel. At 1.4 GHz and 408 MHz the phase forms
smooth fringes the pixels resolve; at 50 and 30 MHz neighbouring tiles have
unrelated phases.

### 4c. Zoom into a few pixels: what a pixel stores vs what it should hold

A map pixel should hold the sky *averaged over its area*; the pixel sum uses
the rotated polarization at the pixel's *one* RM. To see the sky inside a
pixel we need RM between pixel centres: we interpolate it linearly
(`hp.get_interp_val`) and draw it at 1.6″, the pixel size of nside 131072.
The pixel average is the mean over the $4^8 = 65536$ nside-131072 pixels
inside each nside-512 pixel. The spot is chosen where the RM step between
pixels is close to the sky median (0.65 rad m$^{-2}$), so it is typical.

In [ ]:
ZOOM_L, ZOOM_B = 250.049, -16.179  # inside the Fig. 4b patch
ZOOM_ARCMIN, ZOOM_NX = 15.0, 600
Z_FREQS = (408.0, 30.0)


def pixel_average_phasor(pix_ring, k_sub, freqs):
    """<exp(2i lam^2 RM)> over each pixel, RM bilinear on 4**k sub-pixels."""
    ns_sub, nsub = NSIDE * 2**k_sub, 4**k_sub
    nest = hp.ring2nest(NSIDE, pix_ring)
    out = {f: np.empty(pix_ring.size, complex) for f in freqs}
    for idx in np.array_split(np.arange(pix_ring.size),
                              max(1, pix_ring.size * nsub // 2_000_000)):
        kids = (nest[idx, None] * nsub + np.arange(nsub)).ravel()
        th, ph = hp.pix2ang(ns_sub, kids, nest=True)
        rm_sub = hp.get_interp_val(RM, th, ph).reshape(idx.size, nsub)
        for f in freqs:
            out[f][idx] = np.exp(2j * lam2(f) * rm_sub).mean(axis=1)
    return out


zproj = hp.projector.GnomonicProj(rot=(ZOOM_L, ZOOM_B), xsize=ZOOM_NX,
                                  reso=ZOOM_ARCMIN / ZOOM_NX)
zext = zproj.get_extent()
zx, zy = zproj.ij2xy()
zvec = [v.ravel() for v in zproj.xy2vec(zx, zy)]
zth, zph = hp.vec2ang(np.stack(zvec, axis=1))
rm_fine = hp.get_interp_val(RM, zth, zph).reshape(zx.shape)
zpix = hp.vec2pix(NSIDE, *zvec).reshape(zx.shape)
zup, zinv = np.unique(zpix, return_inverse=True)
zinv = zinv.reshape(zx.shape)

# the pixel average must itself be converged in the sub-sampling
for k in (6, 7, 8):
    a = pixel_average_phasor(zup, k, (30.0,))[30.0]
    print(f"4**{k} sub-pixels: rms |pixel average| at 30 MHz = "
          f"{np.sqrt(np.mean(np.abs(a)**2)):.4f}")
zavg = pixel_average_phasor(zup, 8, Z_FREQS)
print("RM step to neighbours here:",
      np.round(step_rm[zup], 2), "rad/m^2")

def plot_zoom(outlines=True):
    fig, axes = plt.subplots(2, 3, figsize=(11.5, 8.2),
                             constrained_layout=True)
    q_norm = Normalize(-1, 1)
    cols = ("sky at 1.6″", "averaged over each pixel",
            "one value per pixel\n(the pixel sum)")
    for r, f0 in enumerate(Z_FREQS):
        maps = (
            np.cos(2 * lam2(f0) * rm_fine),
            zavg[f0].real[zinv],
            np.cos(2 * lam2(f0) * RM[zpix]),
        )
        for c, m in enumerate(maps):
            ax = axes[r, c]
            ax.imshow(m, origin="lower", extent=zext, cmap="RdBu_r",
                      norm=q_norm, interpolation="nearest")
            if outlines:
                for p in zup:
                    b = hp.boundaries(NSIDE, p, step=4)
                    bx, by = zproj.vec2xy(*b)
                    ax.plot(np.r_[bx, bx[:1]], np.r_[by, by[:1]],
                            color="k", lw=0.8)
            ax.set_xlim(zext[:2])
            ax.set_ylim(zext[2:])
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c])
        step = 2 * lam2(f0) * np.median(step_rm[zup])
        step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
        axes[r, 0].set_ylabel(f"{f0:.0f} MHz\n{step_txt} rad per pixel")
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=q_norm, cmap="RdBu_r"),
                      ax=axes, orientation="horizontal", fraction=0.05,
                      pad=0.03, aspect=50)
    cb.set_label("rotated $Q$ / unrotated $Q$")
    return fig


save(plot_zoom(), "04c_pixel_zoom")

**Figure 4c.** A 15′ zoom at $(l, b) = (250.05^\circ, -16.18^\circ)$, inside
the Fig. 4b patch, where the RM step between neighbouring pixels is close to
the sky median. Black diamonds are nside-512 pixels (~6.9′). The sky is uniformly polarized in $Q$ ($U = 0$) to
isolate the Faraday factor, so the colour is rotated over unrotated $Q$.
Left: the rotated $Q$ drawn at 1.6″ (nside-131072 resolution), with RM
interpolated linearly between pixel centres; the kinks in the fringes are
where the interpolation switches neighbours. Middle: that field averaged over
each nside-512 pixel, which is what a map pixel should hold. Right: the
rotated $Q$ at each pixel's single RM, which is what the pixel sum used. At
408 MHz (~0.7 rad per pixel) all three agree. At 30 MHz (~130 rad, ~20 turns
per pixel) the fringes average to nearly zero inside every pixel, but the
pixel sum keeps one point from each at full amplitude with an arbitrary sign.
Physical depolarization makes the map *fainter*; the pixel sum makes it
*scrambled at full brightness*. The interpolated sky is itself a model: the
real sky inside a pixel has RM structure the map does not contain.

### 4d. The whole sky: Fig. 2 with the pixel average

The 4c average for every nside-512 pixel: each pixel is split into
$4^7 = 16384$ sub-pixels (nside 65536), averaged, and discarded, so the large
map is never stored. About 20 min on 8 cores; cached under
`generated_data/cache/` (git-ignored).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

K_FULL = 7
CACHE = REPO / "generated_data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
avg_path = CACHE / f"slides_pixavg_phasor_k{K_FULL}_{F_MAP:.0f}MHz.npy"


def _avg_block(lohi):
    lo, hi = lohi
    return pixel_average_phasor(np.arange(lo, hi), K_FULL, (F_MAP,))[F_MAP]


if avg_path.exists():
    phasor_avg = np.load(avg_path)
else:
    bounds = np.linspace(0, RM.size, 129).astype(int)
    with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as ex:
        parts = list(ex.map(_avg_block, zip(bounds[:-1], bounds[1:])))
    phasor_avg = np.concatenate(parts)
    np.save(avg_path, phasor_avg)
print(f"|<phasor>| over pixels at {F_MAP:.0f} MHz: median "
      f"{np.median(np.abs(phasor_avg)):.4f}, p99 "
      f"{np.percentile(np.abs(phasor_avg), 99):.4f}")

In [ ]:
q, u, qr, ur = qu_rot_maps(F_MAP)
p_avg = (q + 1j * u) * phasor_avg
panels = {
    (0, 0): (q, f"$Q$ at {F_MAP:.0f} MHz"),
    (0, 1): (qr, f"$Q$ rotated at nside {NSIDE}"),
    (0, 2): (p_avg.real, f"$Q$ rotated at nside {NSIDE * 2**K_FULL},\n"
                         f"averaged to nside {NSIDE}"),
}
W_IN, H_IN, MAP_W_IN, GAP_IN = 13.0, 3.4, 4.0, 0.3
MAP_H_IN = MAP_W_IN / 2
fig = plt.figure(figsize=(W_IN, H_IN))
x0 = (W_IN - 3 * MAP_W_IN - 2 * GAP_IN) / 2
t = np.linspace(0, 2 * np.pi, 400)
norm = NORMS["linear"]
for (r, c), (m, title) in panels.items():
    ax = fig.add_axes([(x0 + c * (MAP_W_IN + GAP_IN)) / W_IN,
                       (H_IN - 0.75 - MAP_H_IN) / H_IN,
                       MAP_W_IN / W_IN, MAP_H_IN / H_IN])
    # the pixel-averaged map is ~30x fainter: give it its own scale
    own = c == 2
    if own:
        vown = np.percentile(np.abs(m), 99) / KK
        pnorm = Normalize(-vown, vown)
    img, ext = moll_image(m / KK)
    ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r",
              norm=pnorm if own else norm, interpolation="nearest")
    ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
    ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
    ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
    ax.set_axis_off()
    ax.set_title(title, fontsize=13)
    print(f"{title.replace(chr(10), ' '):55s} rms {m.std()/KK:6.2f}e3 K")
# shared scale under the first two maps, own scale under the third
cax = fig.add_axes([(x0 + 0.6) / W_IN, 0.2 / H_IN,
                    (2 * MAP_W_IN + GAP_IN - 1.2) / W_IN, 0.15 / H_IN])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="RdBu_r"),
                  cax=cax, orientation="horizontal")
cb.set_label("brightness temperature  [$10^3$ K]")
cax2 = fig.add_axes([(x0 + 2 * (MAP_W_IN + GAP_IN) + 0.6) / W_IN,
                     0.2 / H_IN, (MAP_W_IN - 1.2) / W_IN, 0.15 / H_IN])
cb2 = fig.colorbar(plt.cm.ScalarMappable(norm=pnorm, cmap="RdBu_r"),
                   cax=cax2, orientation="horizontal")
cb2.set_label("brightness temperature  [$10^3$ K]")
save(fig, "04d_fullsky_point_vs_average")

**Figure 4d.** WMAP K $Q$ at 30 MHz (as Fig. 2), native nside 512. Left:
intrinsic. Middle: rotated with each pixel's single RM, as the pixel sum did
(Fig. 2). Right: rotated and averaged over each pixel, with RM interpolated
linearly between pixel centres and sampled on 16384 sub-pixels per pixel;
the intrinsic $Q, U$ are held at their pixel values. Left and middle share
Fig. 2's linear scale; the right panel has its own, clipped at the 99th
percentile of $|Q|$, because on the shared scale it is blank. Averaging inside the pixel removes nearly all of the rotated
polarization; the pixel sum kept it at full brightness with scrambled signs.
In the Galactic plane, where the phase turns by thousands of radians per
pixel, even 16384 sub-pixels undersample it, so the value there is itself a
random-walk residual of order 1% and an upper bound. The interpolated RM is a
model: the real sky inside a pixel has RM structure the map does not
contain.

### 4e. Why a wide beam suddenly cares about small scales

A wide beam is a low-pass filter: it reads only the low-$\ell$ content of the
sky it looks at. Faraday rotation moves polarized power from low $\ell$ to
the fringe scale. The angular power spectra of the three maps of Fig. 4d
show where the power sits, and what is left where the beam reads.

In [ ]:
ELL_BEAM = 30  # production arm's beam lmax (docs/measurement-model.md)
zeros = np.zeros_like(q)
spectra = {
    "intrinsic": (q, u),
    "rotated, one value per pixel": (qr, ur),
    "rotated, averaged over each pixel": (p_avg.real, p_avg.imag),
}
cl = {}
for name, (mq, mu) in spectra.items():
    c = hp.anafast([zeros, mq, mu], pol=True)
    cl[name] = c[1] + c[2]  # EE + BB, K^2
ell = np.arange(cl["intrinsic"].size)

# random signs from pixel to pixel: white, C_l = <|P|^2> * pixel area
floor = np.mean(qr**2 + ur**2) * hp.nside2pixarea(NSIDE)

fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
ax.axvspan(2, ELL_BEAM, color="0.9", zorder=0)
ax.text(np.sqrt(2 * ELL_BEAM), 0.5, "LuSEE beam\nreads $\\ell \\lesssim 30$",
        transform=ax.get_xaxis_transform(), ha="center", color="0.35",
        fontsize=12)
for (name, c), col in zip(cl.items(), ("0.2", C_U, C_Q)):
    ax.loglog(ell[2:], c[2:], color=col, lw=1.6, label=name)
ax.axhline(floor, color=C_U, ls="--", lw=1,
           label="random signs per pixel: $\\langle|P|^2\\rangle\\,\\Omega_\\mathrm{pix}$")
ax.set_xlim(2, ell[-1])
ax.set_xlabel("multipole $\\ell$")
ax.set_ylabel(f"$C_\\ell^{{EE}} + C_\\ell^{{BB}}$ at {F_MAP:.0f} MHz  [K$^2$]")
ax.legend(fontsize=11, loc="upper right")
for name, c in cl.items():
    print(f"{name:36s} mean C_l over 2<=l<=30: {c[2:ELL_BEAM+1].mean():.3e} K^2")
print(f"{'white floor':36s} {floor:.3e} K^2")
save(fig, "04e_cl_beam_window")

**Figure 4e.** Polarized angular power spectrum ($C_\ell^{EE} + C_\ell^{BB}$,
`hp.anafast`, nside 512) at 30 MHz of the three maps of Fig. 4d: WMAP K $Q, U$
scaled to 30 MHz (black), rotated with one RM per pixel as the pixel sum did
(orange), and rotated then averaged over each pixel (blue). Grey band: the
multipoles the LuSEE beam reads; the production arm truncates the beam at
$\ell = 30$ (raising it to 48 changes the I-only products by $8\times10^{-5}$;
the sharp horizon gives the polarized kernels a weak high-$\ell$ tail). The
intrinsic sky puts its power at low $\ell$, where the beam reads; its
high-$\ell$ flattening is WMAP's map noise. Rotation as the pixel sum did it
gives a flat spectrum at the white-noise level of pixels with random signs
(dashed): what the beam reads from it is set by the pixel count. Averaging
inside each pixel removes most of that power at every $\ell$, including where
the beam reads. The beam is still a low-pass filter; rotation emptied the
low-$\ell$ band, so what is left there is built from structure below the
pixel scale.

### 4f. The ensemble model's intrinsic spectrum vs WMAP K

The depth-depolarization branch draws the intrinsic polarized sky as a
Gaussian spin-2 field with $C_\ell \propto \exp[-\ell(\ell+1)/\ell_s^2]$,
$\ell_s = 8$ (`scripts/faraday_covariance.py:intrinsic_spectra`). Compare its
shape with WMAP K (the black curve of Fig. 4e), after removing WMAP's white
noise, with both normalized to the same power where the beam reads.

In [ ]:
cl_w = cl["intrinsic"]
noise_w = np.median(cl_w[1000:1500])  # flat high-ell floor: WMAP map noise
sig_w = cl_w - noise_w
fit = slice(2, 101)  # signal-dominated, inside WMAP K's 0.9 deg beam
slope, icpt = np.polyfit(np.log(ell[fit]), np.log(sig_w[fit]), 1)
print(f"WMAP K signal: C_l ~ l^{slope:.2f} over 2 <= l <= 100; "
      f"noise floor {noise_w:.0f} K^2")

band = slice(2, ELL_BEAM + 1)
w = (2 * ell + 1) / (4 * np.pi)


def model_cl(ell_s):
    c = np.exp(-ell * (ell + 1) / ell_s**2)
    c[:2] = 0
    return c


fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
ax.axvspan(2, ELL_BEAM, color="0.9", zorder=0)
ref = np.sum(w[band] * sig_w[band])
ax.loglog(ell[2:300], sig_w[2:300], color="0.2", lw=1.6,
          label="WMAP K, noise removed")
ax.loglog(ell[2:300], np.exp(icpt) * ell[2:300] ** slope, color="0.2",
          ls=":", lw=1.2, label=f"power law, $\\ell^{{{slope:.1f}}}$")
for ell_s, col in ((4, "#9ec5fe"), (8, C_Q), (16, "#08306b")):
    c = model_cl(ell_s)
    c *= ref / np.sum(w[band] * c[band])
    ax.loglog(ell[2:300], c[2:300], color=col, lw=2 if ell_s == 8 else 1.2,
              label=f"model, $\\ell_s = {ell_s}$" + (" (used)" if ell_s == 8 else ""))
    frac = np.sum(w[band] * c[band]) / np.sum(w[2:] * c[2:])
    print(f"model l_s={ell_s:2d}: {100 * frac:5.1f}% of its power at l <= {ELL_BEAM}")
frac_w = np.sum(w[band] * sig_w[band]) / np.sum(w[2:101] * sig_w[2:101])
print(f"WMAP K signal: {100 * frac_w:5.1f}% of its 2 <= l <= 100 power at l <= {ELL_BEAM}")
ax.set_ylim(sig_w[2:300][sig_w[2:300] > 0].min() / 3, sig_w[2] * 5)
ax.set_xlim(2, 300)
ax.set_xlabel("multipole $\\ell$")
ax.set_ylabel(f"$C_\\ell^{{EE}} + C_\\ell^{{BB}}$ at {F_MAP:.0f} MHz  [K$^2$]")
ax.legend(fontsize=11, loc="lower left")
save(fig, "04f_intrinsic_spectrum_vs_wmap")

**Figure 4f.** Polarized angular power spectrum of WMAP 9-yr K $Q, U$ scaled
to 30 MHz ($\beta = -2.8$), with its white noise floor (median $C_\ell$ over
$1000 \le \ell \le 1500$) subtracted, and a power-law fit over
$2 \le \ell \le 100$ (above that WMAP's 0.9° beam suppresses the signal).
Blue: the ensemble model's intrinsic spectrum for $\ell_s = 4, 8, 16$
(8 is used), each scaled to WMAP's power over $\ell \le 30$ (grey), so only
shapes are compared. The model puts nearly all its power in the band the
beam reads; WMAP's spectrum is a power law with power at all $\ell$. In the
strong-rotation limit the speckle depends on the total intrinsic variance,
not its angular shape, so the shape matters mainly for the smooth component
and for how a total-RMS normalization splits between the two.

## 5. The pixel sum does not converge

Stored results of the 2026-08-18 audit (tag `audit-2026-08-18`): test 06,
`shuffle_control.py` (`audit/generated/shuffle_results.json`), and test 07,
`where_it_breaks.py` (values as plotted by `audit/scripts/make_fig.py`). Both
use a $\cos^2\theta$ lobe about a fixed Galactic axis, not the LuSEE beam,
at a single time; the diagnostic compares each measurement against the same
weights, so it does not depend on the beam shape.

In [ ]:
# test 06: band-averaged |P|^2 (total delay power, by Parseval), 30 MHz
NS_AUDIT = np.array([512, 1024, 2048])
POW = {
    ("actual RM map", "real"): [0.5671, 0.1242, 0.02522],
    ("actual RM map", "shuffled"): [0.9196, 0.1876, 0.03052],
    ("RM × 0.02", "real"): [1.3512, 1.0309, 0.9732],
    ("RM × 0.02", "shuffled"): [0.2774, 0.02446, 0.004481],
}
# test 07: |P|_Faraday / |P|_no Faraday vs lambda^2
SWEEP_L2 = np.array([0.01, 0.05, 0.14, 0.56, 2.25, 8.99, 35.95, 99.86,
                     898.76])
SWEEP = {
    512: [8.047e-1, 4.839e-1, 2.688e-1, 2.755e-2, 1.017e-2, 5.266e-3,
          3.145e-3, 8.448e-3, 3.953e-3],
    1024: [8.047e-1, 4.846e-1, 2.670e-1, 3.104e-2, 7.922e-3, 6.602e-3,
           2.612e-3, 2.432e-3, 3.188e-3],
    2048: [8.047e-1, 4.845e-1, 2.665e-1, 3.083e-2, 6.776e-3, 3.726e-3,
           3.672e-3, 2.379e-3, 1.298e-3],
}

fig, (axB, axC) = plt.subplots(1, 2, figsize=(12.0, 4.8),
                               constrained_layout=True)
for (name, kind), y in POW.items():
    col = C_Q if name.startswith("actual") else C_U
    real = kind == "real"
    axB.loglog(NS_AUDIT, y, "o-" if real else "s:", color=col,
               lw=2.2 if real else 1.4, ms=7 if real else 5,
               alpha=1 if real else 0.6, label=f"{name}, {kind}")
guide = POW[("actual RM map", "real")][0] * (NS_AUDIT / 512.0) ** -2
axB.loglog(NS_AUDIT, guide, color="0.6", lw=1, ls=(0, (1, 2)))
axB.text(1100, guide[1] * 2.2, "$\\propto 1/N_\\mathrm{pix}$", color="0.45")
axB.set_xticks(NS_AUDIT, [str(n) for n in NS_AUDIT])
axB.minorticks_off()
axB.set_xlabel("nside")
axB.set_ylabel("band-averaged $|P|^2$ at 30 MHz  [arb.]")
axB.legend(fontsize=10.5, loc="lower left")

axC.axvspan(lam2(50.0), lam2(10.0), color="0.92", zorder=0)
axC.text(np.sqrt(lam2(50.0) * lam2(10.0)), 0.35, "LuSEE\n50 → 10 MHz",
         ha="center", color="0.4", fontsize=11)
for n, mk, col in ((512, "^", "#9ec5fe"), (1024, "s", "#5b9bf8"),
                   (2048, "o", C_Q)):
    axC.loglog(SWEEP_L2, SWEEP[n], mk + "-", color=col, lw=1.8, ms=6,
               label=f"nside {n}")
axC.set_xlabel("$\\lambda^2$  [m$^2$]")
axC.set_ylabel("$|P|_\\mathrm{Faraday}\\,/\\,|P|_\\mathrm{no\\ Faraday}$")
top = axC.secondary_xaxis(
    "top", functions=(lambda l2: C / np.sqrt(l2) / 1e6,
                      lambda f: (C / (np.maximum(f, 1e-3) * 1e6)) ** 2))
top.set_xticks([3000, 1000, 300, 100, 30, 10],
               ["3000", "1000", "300", "100", "30", "10"])
top.set_xlabel("frequency  [MHz]")
axC.legend(fontsize=10.5, loc="lower left")
save(fig, "05_convergence")

**Figure 5.** Convergence of the beam-weighted pixel sum, from the
2026-08-18 audit ($\cos^2\theta$ beam about a fixed axis, one time step,
WMAP K $Q, U$ and the Hutschenreuter RM map synthesised to $\ell_\max = 1023$ so
that refining nside changes only the sampling). Left: band-averaged $|P|^2$
across the ±0.1 MHz band at 30 MHz against nside. With the real RM map it
falls 4× per nside doubling ($\propto 1/N_\mathrm{pix}$), the signature of a
random walk, and shuffling the RM values over the sky barely changes it. The
control, RM × 0.02, whose phase the grid resolves, levels off and is 200×
above its own shuffle at nside 2048: the method converges when the phase is
sampled. Right: beam-integrated $|P|$ with Faraday rotation over $|P|$ without,
against $\lambda^2$. The three resolutions agree down to $\lambda^2 \approx
1$ m$^2$ (~300 MHz) and separate below, throughout the LuSEE band (shaded).
A rigid rotation of the grid at nside 1024, which changes no physics, changes
$|P|$ at 30 MHz by 7.2×.